In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, accuracy_score, roc_auc_score
)
from itertools import product
import warnings
warnings.filterwarnings('ignore')
from tqdm.auto import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
%cd /content/drive/MyDrive/DSCI-691-Project/

/content/drive/.shortcut-targets-by-id/178re1vXBOn1poHWHRN9tiWVeMNB2tYhp/DSCI-691-Project


In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")

Device: cuda


# 1. dataset G

## 1.1 Model definitions

In [ ]:
# model definitions

class CNN(nn.Module):
    def __init__(self, seq_len=300):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=11, padding=5), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=7, padding=3), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.head(self.conv(x))


class BiLSTM(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.lstm = nn.LSTM(4, hidden, num_layers=1, batch_first=True,
                            bidirectional=True)
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(hidden * 2, 64), nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        pooled = out.mean(dim=1)
        return self.head(pooled)



def kmer_features(seqs, k=3):
  kmer_vocab = [''.join(p) for p in product('ACGT', repeat=k)]
  kmer_idx = {k: i for i, k in enumerate(kmer_vocab)}
  X = np.zeros((len(seqs), len(kmer_vocab)), dtype=np.float32)
  for r, seq in enumerate(seqs):
    seq = seq.upper()
    total = 0
    for i in range(len(seq) - k + 1):
      km = seq[i:i + k]
      if km in kmer_idx:
        X[r, kmer_idx[km]] += 1
        total += 1
    if total > 0:
      X[r] /= total
  return X


In [ ]:
# training scripts
def train_nn(model, train_loader, val_loader):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.CrossEntropyLoss(weight=cw_t)
    best_val, best_state = -1, None
    for epoch in range(NN_EPOCHS):
        model.train()
        for Xb, yb in tqdm(train_loader, leave=False):
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            opt.step()
        # validation macro F1 for model selection
        vy, vp, _ = predict_nn(model, val_loader)
        vf1 = f1_score(vy, vp, average='macro')
        print(vf1)
        if vf1 > best_val:
            best_val, best_state = vf1, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    print(f"best val macro F1 = {best_val:.4f}")
    return model

def predict_nn(model, loader):
    model.eval()
    ys, ps, probs = [], [], []
    with torch.no_grad():
        for Xb, yb in tqdm(loader, leave=False):
            Xb = Xb.to(device)
            logits = model(Xb)
            prob = torch.softmax(logits, dim=1)[:, 1]
            ps.extend(logits.argmax(1).cpu().numpy())
            probs.extend(prob.cpu().numpy())
            ys.extend(yb.numpy())
    return np.array(ys), np.array(ps), np.array(probs)

## 1.2 Utils

In [ ]:
def record(model_name, split_label, y_true, y_pred, y_prob=None):
    """Append one result row and print it."""
    row = {
        'Model': model_name,
        'Split': split_label,
        'f1': f1_score(y_true, y_pred, average='macro'),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
    }
    results.append(row)
    print(f"  {model_name:<16} {split_label:<10} "
          f"F1={row['f1']:.4f}  BalAcc={row['balanced_accuracy']:.4f}  "
          f"AUC={row['roc_auc']:.4f}" if y_prob is not None
          else f"  {model_name:<16} {split_label:<10} "
               f"F1={row['f1']:.4f}  BalAcc={row['balanced_accuracy']:.4f}")
    return row


BASES = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

def one_hot(seqs, length=300):
    X = np.zeros((len(seqs), length, 4), dtype=np.float32)
    for r, seq in enumerate(seqs):
        for i, b in enumerate(seq.upper()[:length]):
            if b in BASES:
                X[r, i, BASES[b]] = 1.0
    return X

def make_loader(ds, batch=64, shuffle=False):
    X = one_hot(ds['sequence'].tolist())
    y = ds['label'].values
    X = torch.tensor(X).permute(0, 2, 1)
    y = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(X, y), batch_size=batch, shuffle=shuffle)

## 1.3 Run train and eval

In [ ]:
# set globals
KMER_K = 3 # only for knn
NN_EPOCHS = 15

In [ ]:
#grab data
paths_dict = {
    'train': 'dataset_g/train.csv',
    'test-heldout': 'dataset_g/test_OOD.csv',
    'test-id': 'dataset_g/test_ID.csv',
    'test-matched_id': 'dataset_g/test_motif_matched_ID.csv'
    }

ds = {}
for fold, path in paths_dict.items():
  df = pd.read_csv(path)
  ds[fold] = df[["sequence","label"]]

# dataset - heldout
train_df, val_df = train_test_split(ds['train'], test_size=0.2, random_state=42)

ds['train'] = train_df
ds['val'] = val_df


In [ ]:
# train data loader
results = []
train_loader = make_loader(ds['train'], shuffle=True)
val_loader = make_loader(ds['val'], shuffle=True)

# Class weights from training split
y_train = ds['train']['label'].values
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
cw_t = torch.tensor(cw, dtype=torch.float).to(device)


# train models
# knn
print("training KNN")
X_tr = kmer_features(ds['train']['sequence'].tolist())
scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr)

knn = KNeighborsClassifier(n_neighbors=15, weights='distance')
knn.fit(X_tr, y_train)

#cnn
print("training CNN")
cnn = train_nn(CNN(), train_loader, val_loader)

#lstm
print("training BiLSTM")
bilstm = train_nn(BiLSTM(), train_loader, val_loader)


print("Begin eval...")
# eval models
for test in ['test-heldout', 'test-id', 'test-matched_id']:
  #knn
  print(f"Evaluating KNN for {test}")
  Xs = scaler.transform(kmer_features(ds[test]['sequence'].tolist()))
  ys = ds[test]['label'].values
  pred = knn.predict(Xs)
  prob = knn.predict_proba(Xs)[:, 1]
  record(f'k-mer+KNN (k={KMER_K})', test, ys, pred, prob)

  # NN models
  test_loader = make_loader(ds[test], shuffle=False)

  y, p, pr = predict_nn(cnn, test_loader)
  record('CNN', test, y, p, pr)

  y, p, pr = predict_nn(bilstm, test_loader)
  record('BiLSTM', test, y, p, pr)

training KNN
training CNN


  0%|          | 0/246 [00:00<?, ?it/s]

  0%|          | 0/62 [00:00<?, ?it/s]

0.7025182368718871


  0%|          | 0/246 [00:00<?, ?it/s]

...

  BiLSTM           test-id    F1=0.7515  BalAcc=0.7408  AUC=0.8084
Evaluating KNN for test-matched_id
  k-mer+KNN (k=3)  test-matched_id F1=0.7533  BalAcc=0.7991  AUC=0.9631


  0%|          | 0/25 [00:00<?, ?it/s]

  CNN              test-matched_id F1=0.9131  BalAcc=0.9094  AUC=0.9620


  0%|          | 0/25 [00:00<?, ?it/s]

  BiLSTM           test-matched_id F1=0.7766  BalAcc=0.7846  AUC=0.8488


# 2 dataset M

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_PATH = "."
DATASET_PATH = f"{BASE_PATH}/dataset_m/promoter_final_dataset.csv"
OUT_CSV = f"{BASE_PATH}/baseline_results.csv"

KMER_K = 3
NN_EPOCHS = 15
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")

Device: cuda


In [ ]:
df = pd.read_csv(DATASET_PATH)
sd = {n: df[df['split'] == n].reset_index(drop=True)
      for n in ['train', 'val', 'id_test', 'ood_test']}
print("\nSplit sizes:")
for n, s in sd.items():
    print(f"  {n}: {len(s)} ({s['label'].mean()*100:.1f}% promoters)")


Split sizes:
  train: 12021 (24.7% promoters)
  val: 2576 (24.7% promoters)
  id_test: 2577 (24.7% promoters)
  ood_test: 3726 (24.7% promoters)


In [ ]:
SEQ_LEN = len(df.iloc[0]['sequence'])
print(f"Sequence length: {SEQ_LEN} bp")

# Class weights from training split (same as DNABERT script)
y_train = sd['train']['label'].values
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
cw_t = torch.tensor(cw, dtype=torch.float).to(device)
print(f"Class weights: {cw.tolist()}")

Sequence length: 300 bp
Class weights: [0.6638502319416832, 2.025783619817998]


In [ ]:
results = []
def record(model_name, split_label, y_true, y_pred, y_prob=None):
    """Append one result row and print it."""
    row = {
        'Model': model_name,
        'Split': split_label,
        'f1': f1_score(y_true, y_pred, average='macro'),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
    }
    results.append(row)
    print(f"  {model_name:<16} {split_label:<10} "
          f"F1={row['f1']:.4f}  BalAcc={row['balanced_accuracy']:.4f}  "
          f"AUC={row['roc_auc']:.4f}" if y_prob is not None
          else f"  {model_name:<16} {split_label:<10} "
               f"F1={row['f1']:.4f}  BalAcc={row['balanced_accuracy']:.4f}")
    return row

In [ ]:
print(f"[1] k-mer (k={KMER_K}) frequency + KNN:")

kmer_vocab = [''.join(p) for p in product('ACGT', repeat=KMER_K)]
kmer_idx = {k: i for i, k in enumerate(kmer_vocab)}

def kmer_features(seqs, k=KMER_K):
    X = np.zeros((len(seqs), len(kmer_vocab)), dtype=np.float32)
    for r, seq in enumerate(seqs):
        seq = seq.upper()
        total = 0
        for i in range(len(seq) - k + 1):
            km = seq[i:i + k]
            if km in kmer_idx:
                X[r, kmer_idx[km]] += 1
                total += 1
        if total > 0:
            X[r] /= total
    return X

X_tr = kmer_features(sd['train']['sequence'].tolist())
scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr)

knn = KNeighborsClassifier(n_neighbors=15, weights='distance')
knn.fit(X_tr, y_train)

for split_key, split_label in [('id_test', 'ID Test'), ('ood_test', 'OOD Test')]:
    Xs = scaler.transform(kmer_features(sd[split_key]['sequence'].tolist()))
    ys = sd[split_key]['label'].values
    pred = knn.predict(Xs)
    prob = knn.predict_proba(Xs)[:, 1]
    record(f'k-mer+KNN (k={KMER_K})', split_label, ys, pred, prob)

BASES = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

def one_hot(seqs, length=SEQ_LEN):
    X = np.zeros((len(seqs), length, 4), dtype=np.float32)
    for r, seq in enumerate(seqs):
        for i, b in enumerate(seq.upper()[:length]):
            if b in BASES:
                X[r, i, BASES[b]] = 1.0
    return X

def make_loader(split_key, batch=64, shuffle=False):
    X = one_hot(sd[split_key]['sequence'].tolist())
    y = sd[split_key]['label'].values
    X = torch.tensor(X).permute(0, 2, 1)
    y = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(X, y), batch_size=batch, shuffle=shuffle)

train_loader = make_loader('train', shuffle=True)
loaders = {k: make_loader(k) for k in ['val', 'id_test', 'ood_test']}

def train_nn(model, name):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.CrossEntropyLoss(weight=cw_t)
    best_val, best_state = -1, None
    for epoch in range(NN_EPOCHS):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            opt.step()
        # validation macro F1 for model selection
        vy, vp, _ = predict_nn(model, loaders['val'])
        vf1 = f1_score(vy, vp, average='macro')
        if vf1 > best_val:
            best_val, best_state = vf1, {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    print(f"  {name}: best val macro F1 = {best_val:.4f}")
    return model

def predict_nn(model, loader):
    model.eval()
    ys, ps, probs = [], [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            logits = model(Xb)
            prob = torch.softmax(logits, dim=1)[:, 1]
            ps.extend(logits.argmax(1).cpu().numpy())
            probs.extend(prob.cpu().numpy())
            ys.extend(yb.numpy())
    return np.array(ys), np.array(ps), np.array(probs)

[1] k-mer (k=3) frequency + KNN:
  k-mer+KNN (k=3)  ID Test    F1=0.6836  BalAcc=0.6564  AUC=0.7936
  k-mer+KNN (k=3)  OOD Test   F1=0.7650  BalAcc=0.8227  AUC=0.9082


In [ ]:
print("[2] CNN:")

class CNN(nn.Module):
    def __init__(self, seq_len=SEQ_LEN):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=11, padding=5), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=7, padding=3), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.head(self.conv(x))

cnn = train_nn(CNN(), 'CNN')
for split_key, split_label in [('id_test', 'ID Test'), ('ood_test', 'OOD Test')]:
    y, p, pr = predict_nn(cnn, loaders[split_key])
    record('CNN', split_label, y, p, pr)

[2] CNN:
  CNN: best val macro F1 = 0.7616
  CNN              ID Test    F1=0.7483  BalAcc=0.7551  AUC=0.8376
  CNN              OOD Test   F1=0.5979  BalAcc=0.7294  AUC=0.9294


In [ ]:
print("[3] BiLSTM:")

class BiLSTM(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.lstm = nn.LSTM(4, hidden, num_layers=1, batch_first=True,
                            bidirectional=True)
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(hidden * 2, 64), nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        pooled = out.mean(dim=1)
        return self.head(pooled)

bilstm = train_nn(BiLSTM(), 'BiLSTM')
for split_key, split_label in [('id_test', 'ID Test'), ('ood_test', 'OOD Test')]:
    y, p, pr = predict_nn(bilstm, loaders[split_key])
    record('BiLSTM', split_label, y, p, pr)

[3] BiLSTM:
  BiLSTM: best val macro F1 = 0.6831
  BiLSTM           ID Test    F1=0.6778  BalAcc=0.6772  AUC=0.7447
  BiLSTM           OOD Test   F1=0.5118  BalAcc=0.6612  AUC=0.8875


In [ ]:
out = pd.DataFrame(results)
out.to_csv(OUT_CSV, index=False)
print(out.to_string(index=False))
print(f"\nSaved: {OUT_CSV}")

          Model    Split       f1  balanced_accuracy  accuracy  roc_auc
k-mer+KNN (k=3)  ID Test 0.683647           0.656403  0.817617 0.793552
k-mer+KNN (k=3) OOD Test 0.765039           0.822719  0.796296 0.908222
            CNN  ID Test 0.748321           0.755147  0.807916 0.837568
            CNN OOD Test 0.597895           0.729401  0.603865 0.929445
         BiLSTM  ID Test 0.677848           0.677193  0.761350 0.744697
         BiLSTM OOD Test 0.511758           0.661153  0.512614 0.887482

Saved: ./baseline_results.csv
